# 03 · Backtest

Aplicação do sinal nas 64 datas de decisão (ago/2020 a nov/2025), com custos e referências. Não mede desempenho: isso é do 04.

- [x] Lê sinais, filtro e retornos do 02
- [x] Reproduz a série de referência do código antigo
- [x] Carteiras A a E e referências (Ibovespa, CDI, universo igual)
- [x] Casos de saída da bolsa
- [x] Sensibilidades (tamanho, corte de saída, defasagem)
- [x] Giro e custos conferidos
- [x] Grava `backtest.pkl` e os CSVs

## 1. Entradas e parâmetros
Carrega as tabelas do 02, confere o alinhamento e define as 64 datas de decisão.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

PASTA = Path('../data/processed')
PASTA_TABELAS = Path('../results/tabelas')
REFERENCIA = Path('../docs/referencia_serie_E.csv')

FRACAO = 0.20
MINIMO_CARTEIRA = 10
CORTE_SAIDA = -0.30
RETORNO_AUSENTE = 0.0
CUSTO_BPS = 20.0
GRADE_BPS = [0, 5, 10, 15, 20, 30, 50]
SPREAD_REFERENCIA = 0.05
LIMITE_ALAVANCAGEM = 3.0

NOMES = ['retorno_mensal', 'elegivel', 'aprovada', 'sinal_residual', 'sinal_residual_legado', 'sinal_bruto', 'saidas', 'cdi_mensal', 'ibov_mensal',
         'roic', 'alavancagem', 'contab_ebitda', 'contab_divida_liquida']
dados = {n: pd.read_pickle(PASTA / f'{n}.pkl') for n in NOMES}

retorno = dados['retorno_mensal']
datas = retorno.index
for n in NOMES:
    if n == 'saidas':
        continue
    assert dados[n].index.equals(datas), n
    if dados[n].ndim == 2:
        assert dados[n].columns.equals(retorno.columns), n

elegivel = dados['elegivel']
aprovada = dados['aprovada']
sinal_residual = dados['sinal_residual']
sinal_bruto = dados['sinal_bruto'].where(sinal_residual.notna())
cdi = dados['cdi_mensal']
ibov = dados['ibov_mensal']
ultimo_com_preco = dados['saidas']['ultimo_mes_com_preco']

decisao = datas[sinal_residual.notna().any(axis=1).to_numpy()]
assert decisao[-1] == datas[-2]
print(decisao[0].date(), 'a', decisao[-1].date(), '|', len(decisao), 'datas de decisão')

## 2. Carteira e retorno do mês
`escolher` monta a carteira de uma data (ranking, filtro, N). `retorno_no_mes` aplica o corte de −30% à ação que saiu da bolsa.

In [ ]:
def escolher(t, sinal, usar_filtro, quantidade, aprov):
    ranking = sinal.loc[t][elegivel.loc[t] & sinal.loc[t].notna()].sort_values(ascending=False)
    n_elegiveis = len(ranking)
    if usar_filtro:
        ranking = ranking[aprov.loc[t, ranking.index].to_numpy()]
    if quantidade == 'quintil':
        n = int(np.ceil(FRACAO * n_elegiveis))
    elif quantidade == 'quintil_aprovadas':
        n = max(MINIMO_CARTEIRA, int(np.ceil(FRACAO * len(ranking))))
    elif quantidade == 'todas':
        n = len(ranking)
    else:
        n = int(quantidade)
    escolhidas = ranking.index[:n]
    if len(escolhidas) < MINIMO_CARTEIRA:
        return pd.Series(dtype=float)
    return pd.Series(1 / len(escolhidas), index=escolhidas)


def retorno_no_mes(codigos, s, corte):
    anterior = datas[datas.get_loc(s) - 1]
    valores = retorno.loc[s, codigos].copy()
    faltou = valores.isna()
    saiu = faltou & (ultimo_com_preco.reindex(codigos) == anterior).to_numpy()
    valores[saiu] = corte
    valores[faltou & ~saiu] = RETORNO_AUSENTE
    return valores, int(saiu.sum()), int((faltou & ~saiu).sum())

## 3. Backtest
Para cada data de decisão: escolhe a carteira, mede compras e vendas contra os pesos anteriores (com deriva) e calcula o retorno bruto do mês seguinte.

In [ ]:
def backtest(sinal, usar_filtro, quantidade='quintil', corte=CORTE_SAIDA, defasagem=0, aprov=None):
    aprov = aprovada if aprov is None else aprov
    linhas, pesos = [], {}
    anteriores = pd.Series(dtype=float)
    for t in decisao:
        pos = datas.get_loc(t)
        if pos + 1 + defasagem >= len(datas):
            continue
        s = datas[pos + 1 + defasagem]
        novos = escolher(t, sinal, usar_filtro, quantidade, aprov)
        diferenca = novos.sub(anteriores, fill_value=0.0)
        compras, vendas = diferenca.clip(lower=0).sum(), -diferenca.clip(upper=0).sum()
        if novos.empty:
            bruto, saidas, ausentes = cdi.loc[s], 0, 0
            anteriores = pd.Series(dtype=float)
        else:
            valores, saidas, ausentes = retorno_no_mes(novos.index, s, corte)
            bruto = (novos * valores).sum()
            deriva = novos * (1 + valores)
            anteriores = deriva / deriva.sum()
        linhas.append({'mes': s, 'retorno_bruto': bruto, 'giro': compras + vendas, 'compras': compras, 'vendas': vendas,
                       'numero_acoes': len(novos), 'na_alternativa': int(novos.empty), 'saidas': saidas, 'ausentes': ausentes})
        pesos[s] = novos
    tabela = pd.DataFrame(linhas).set_index('mes')
    pesos = pd.DataFrame(pesos).T.reindex(columns=retorno.columns).fillna(0.0)
    return tabela, pesos


def liquido(tabela, bps):
    return tabela['retorno_bruto'] - bps / 10000 * tabela['giro']

## 4. Série de referência
Roda o E com as regras do código antigo (piso de ROIC com 5 pontos acima do CDI) e compara com `docs/referencia_serie_E.csv`. É teste de tradução, não resultado.

Usa `sinal_residual_legado` (30 de 36), a regra com que a série de referência foi gerada.

In [ ]:
piso = (1 + cdi).rolling(12).apply(np.prod, raw=True) - 1
ebitda, divida = dados['contab_ebitda'], dados['contab_divida_liquida']
aprovada_referencia = (elegivel & ebitda.notna() & divida.notna() & ebitda.gt(0)
                       & ((divida <= 0) | dados['alavancagem'].le(LIMITE_ALAVANCAGEM))
                       & dados['roic'].gt(piso + SPREAD_REFERENCIA, axis=0))

referencia = pd.read_csv(REFERENCIA, parse_dates=['mes']).set_index('mes')
E_referencia, _ = backtest(dados['sinal_residual_legado'], True, aprov=aprovada_referencia)

c = pd.DataFrame({'bruto': E_referencia['retorno_bruto'], 'bruto_ref': referencia['retorno_bruto'],
                  'giro': E_referencia['giro'], 'giro_ref': referencia['giro'],
                  'n': E_referencia['numero_acoes'], 'n_ref': referencia['numero_acoes']})
dif = (c['bruto'] - c['bruto_ref']).abs()
print(len(c), 'meses | iguais (dif < 1e-6):', int((dif < 1e-6).sum()), '| diferença máxima:', round(dif.max(), 4))
print('mesmo número de ações:', int((c['n'] == c['n_ref']).sum()), '| giro: diferença máxima', round((c['giro'] - c['giro_ref']).abs().max(), 4))
print(c[dif >= 1e-6].round(4).to_string())

## 5. Ablação
Roda as cinco versões (A só filtro, B bruto, C bruto com filtro, D residual, E completo) com as regras base.

In [ ]:
versoes = {
    'A_filtro': (sinal_residual, True, 'todas'),
    'B_bruto': (sinal_bruto, False, 'quintil'),
    'C_bruto_filtro': (sinal_bruto, True, 'quintil'),
    'D_residual': (sinal_residual, False, 'quintil'),
    'E_residual_filtro': (sinal_residual, True, 'quintil'),
}
series, pesos = {}, {}
for nome, (sinal, filtro, quantidade) in versoes.items():
    series[nome], pesos[nome] = backtest(sinal, filtro, quantidade)

meses = series['E_residual_filtro'].index
print(len(meses), 'meses |', meses[0].date(), 'a', meses[-1].date())
resumo = pd.DataFrame({n: {'ações (média)': t['numero_acoes'].mean(), 'giro médio': t['giro'].mean(),
                           'compras médias': t['compras'].mean(), 'meses em CDI': t['na_alternativa'].sum(),
                           'saídas': t['saidas'].sum(), 'ausentes': t['ausentes'].sum()} for n, t in series.items()}).T
print(resumo.round(3).to_string())

## 6. Saídas
Lista as ações do E que ficaram sem preço no mês de retorno e como foram tratadas.

In [ ]:
E, pesos_E = series['E_residual_filtro'], pesos['E_residual_filtro']
casos = []
for s in E.index[(E['saidas'] > 0) | (E['ausentes'] > 0)]:
    anterior = datas[datas.get_loc(s) - 1]
    for codigo in pesos_E.columns[pesos_E.loc[s] > 0]:
        if pd.isna(retorno.loc[s, codigo]):
            ultimo = ultimo_com_preco.get(codigo, pd.NaT)
            casos.append({'mes': s.date(), 'codigo': codigo, 'ultimo_preco': ultimo.date() if pd.notna(ultimo) else None,
                          'peso': round(pesos_E.loc[s, codigo], 3), 'tratado_como': 'saída' if ultimo == anterior else 'ausente'})
print(pd.DataFrame(casos).to_string(index=False))

## 7. Referências
Ibovespa, CDI e a média simples das elegíveis com sinal (universo igual), todos brutos, nos mesmos 64 meses.

In [ ]:
def universo_igual():
    saida = {}
    for t in decisao:
        s = datas[datas.get_loc(t) + 1]
        codigos = retorno.columns[(elegivel.loc[t] & sinal_residual.loc[t].notna()).to_numpy()]
        saida[s] = retorno_no_mes(codigos, s, CORTE_SAIDA)[0].mean()
    return pd.Series(saida)

referencias = pd.DataFrame({'Ibovespa': ibov.loc[meses], 'CDI': cdi.loc[meses], 'Universo_igual': universo_igual()})
print(referencias.notna().sum().to_dict())

## 8. Sensibilidades
E com outras regras de tamanho (quintil das aprovadas, 20, 25 e 30 ações), com corte de saída de 0% e −100%, e com defasagem de um mês.

In [ ]:
sensibilidades = {
    'E_quintil_aprovadas': backtest(sinal_residual, True, 'quintil_aprovadas'),
    'E_fixo_20': backtest(sinal_residual, True, 20),
    'E_fixo_25': backtest(sinal_residual, True, 25),
    'E_fixo_30': backtest(sinal_residual, True, 30),
    'E_corte_0': backtest(sinal_residual, True, corte=0.0),
    'E_corte_menos100': backtest(sinal_residual, True, corte=-1.0),
    'E_defasagem_1m': backtest(sinal_residual, True, defasagem=1),
}
tabela_sens = pd.DataFrame({n: {'meses': len(t), 'ações (média)': t['numero_acoes'].mean(), 'giro médio': t['giro'].mean(),
                                'meses em CDI': t['na_alternativa'].sum(), 'saídas': t['saidas'].sum()}
                            for n, (t, _) in sensibilidades.items()}).T
print(tabela_sens.round(3).to_string())

## 9. Conferências
Testes que travam o notebook se algo estiver errado: giro igual a compras mais vendas, pesos somando 1 e só ações elegíveis e aprovadas na carteira.

In [ ]:
anterior_elegivel = elegivel.shift(1, fill_value=False)
anterior_aprovada = aprovada.shift(1, fill_value=False)
for nome, tabela in series.items():
    p = pesos[nome].reindex(tabela.index)
    investido = tabela['na_alternativa'] == 0
    assert (tabela['giro'] - tabela['compras'] - tabela['vendas']).abs().max() < 1e-12, nome
    assert np.isclose(tabela['giro'].iloc[0], 1.0), nome
    assert np.allclose(p[investido].sum(axis=1), 1.0), nome
    assert not ((p > 0) & ~anterior_elegivel.reindex(p.index)).any().any(), nome
    if versoes[nome][1]:
        assert not ((p > 0) & ~anterior_aprovada.reindex(p.index)).any().any(), nome

E = series['E_residual_filtro']
print('giro bilateral médio:', round(E['giro'].mean(), 3), '| compras (um lado):', round(E['compras'].mean(), 3))
print('custo médio por mês a', CUSTO_BPS, 'bps por lado:', round(CUSTO_BPS / 100 * E['giro'].mean(), 3), '% do capital')

## 10. Saída
Grava `backtest.pkl` em `data/processed` e os retornos brutos e o giro em `results/tabelas`.

In [ ]:
PASTA_TABELAS.mkdir(parents=True, exist_ok=True)
resultado = {'series': series, 'pesos': pesos, 'referencias': referencias, 'sensibilidades': sensibilidades,
             'parametros': {'fracao': FRACAO, 'minimo_carteira': MINIMO_CARTEIRA, 'corte_saida': CORTE_SAIDA,
                            'custo_bps': CUSTO_BPS, 'grade_bps': GRADE_BPS}}
pd.to_pickle(resultado, PASTA / 'backtest.pkl')
bruto = pd.DataFrame({n: t['retorno_bruto'] for n, t in series.items()}).join(referencias)
giro = pd.DataFrame({n: t['giro'] for n, t in series.items()})
bruto.to_csv(PASTA_TABELAS / '03_retornos_brutos.csv')
giro.to_csv(PASTA_TABELAS / '03_giro.csv')
print(bruto.shape, giro.shape)

## 11. Resumo 
**Entrega:** `backtest.pkl` com as séries das versões A a E, os pesos, as referências, as sensibilidades e os parâmetros.

**Conferências que passaram:** reprodução de 51 dos 64 meses da série de referência, giro igual a compras mais vendas, pesos somando 1, carteiras só com ações elegíveis e aprovadas.

**Execução:** retorno mensal de fechamento a fechamento, que aproxima o preço de abertura do primeiro pregão seguinte pelo fechamento anterior (decisão de 01/11); o fechamento do primeiro pregão seguinte é sensibilidade opcional.

